In [ ]:
# General libraries
import numpy as np
import pandas as pd
import nibabel as nib
import json
from pathlib import Path
from functools import partial
from tqdm import tqdm
import ants
from scipy.stats import linregress

# Plotting libraries
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import plotly.express as px
import colorcet as cc

# Custom functions
import pp_utils as pp
import calib_utils as cal

In [ ]:
b0_threshold = 5

In [ ]:
def load_data(dir_name, id):

    scans_fname = dir_name / (id + "_" + "RAW_MPPCA_REG.nii.gz")
    param_fname = dir_name / (id + "_" + "params.h5")
    mask_fname = dir_name / (id + "_" + "MASK.nii.gz")

    scans_data = nib.load(scans_fname).get_fdata().astype(np.float32)
    aff = nib.load(scans_fname).affine
    spacing = pp.spacing_from_affine(aff)
    mask = nib.load(mask_fname).get_fdata().astype(int)

    df = pd.read_hdf(param_fname, key="df", mode="r")
    scans_data_reshuffled = np.transpose(scans_data, [3,0,1,2])
    df["image"] = list(scans_data_reshuffled)

    df["adc"] = df["temperature"].apply(partial(pp.water_diffusivity, polynomial=True))

    return df, mask, spacing

In [ ]:
S1_DIR = Path("./calibration")
s1_id = S1_DIR.stem
S1_DERIV_DIR = S1_DIR / "derivatives"

s1_df, s1_mask, s1_spacing = load_data(S1_DERIV_DIR, s1_id)
s1_relevant_df = s1_df[s1_df["dwi_grad_duration"]==0.035]

S2_DIR = Path("./scanner_2")
s2_id = S2_DIR.stem
S2_DERIV_DIR = S2_DIR / "derivatives"

s2_df, s2_mask, s2_spacing = load_data(S2_DERIV_DIR, s2_id)
s2_df["dwi_direction"] = s2_df["seq_name"].apply(lambda x: x[5].upper())

output_dir = S2_DIR / "plots"
output_dir.mkdir(exist_ok=True)

In [ ]:
# Register CNS scans to CRF scans
b0_threshodl = 5
s1s0 = np.mean(s1_df[s1_df["b_value"] < b0_threshodl]["image"].values, axis = 0)
s2s0 = np.mean(s2_df[s2_df["b_value"] < b0_threshodl]["image"].values, axis = 0)

fixed_a = ants.from_numpy(s1s0, spacing=s1_spacing)
moving_a = ants.from_numpy(s2s0, spacing=s2_spacing)

reg = ants.registration(fixed=fixed_a,
                        moving=moving_a,
                        type_of_transform="SyN",
                        flow_sigma=10,
                        total_sigma=10,
                        aff_metric="mattes",
                        syn_metric="mattes",
                        grad_step=0.1)

s2_data = s2_df["image"].values
reg_scan_list = []
for scan in s2_data:
    scan_a = ants.from_numpy(scan, spacing=s2_spacing)
    reg_scan = ants.apply_transforms(fixed=fixed_a,
                                     moving=scan_a,
                                     transformlist=reg["fwdtransforms"],
                                     interpolator="cosineWindowedSinc")
    reg_scan_list.append(reg_scan)

s2_df["reg_image"] = reg_scan_list

s2_mask_reg = ants.apply_transforms(fixed=fixed_a,
                                    moving=ants.from_numpy(s2_mask, spacing=s2_spacing),
                                    transformlist=reg["fwdtransforms"],
                                    interpolator="nearestNeighbor").numpy()

marg = 5 # mm
combo_mask = cal.shrink_mask_mm(s1_mask, s1_spacing, marg).astype(bool) & cal.shrink_mask_mm(s2_mask_reg, s2_spacing, marg).astype(bool)

In [ ]:
def calc_cmaps_df(df, mask, image_col, b0_threshold=5):

    _mask = mask.astype(bool)
    
    calib_parts = []
    
    for axis in tqdm(df["dwi_direction"].unique()):
        
        sub_df = df[df["dwi_direction"]==axis].copy()

        # Ensure scans are in acquisition order
        sub_df = sub_df.sort_values("scan_time").copy()

        # Put the b=0 image on b=0 rows, NaN elsewhere
        sub_df["s0_image"] = sub_df[image_col].where(sub_df["b_value"] < b0_threshold)

        # Forward-fill so each DWI scan gets the most recent preceding b=0 image
        sub_df["s0_image"] = sub_df["s0_image"].ffill()

        # Keep only diffusion-weighted scans
        dwi_df = sub_df[sub_df["b_value"] > b0_threshold].copy()

        # Optional safety check: drop rows that do not have a preceding b=0 scan
        dwi_df = dwi_df[dwi_df["s0_image"].notna()].copy()

        dwi_df["bmap"] = dwi_df.apply(
            lambda row: cal.bmap_3d(
                row["s0_image"],
                row[image_col],
                row["adc"],
                mask=_mask
            ),
            axis=1
        )
        dwi_df["cmap"] = dwi_df["bmap"] / dwi_df["b_value"]

        calib_parts.append(dwi_df)
        
    calib_df = pd.concat(calib_parts, axis=0, ignore_index=False)
    return calib_df

In [ ]:
s1_calib_df = calc_cmaps_df(s1_relevant_df, mask=s1_mask,  image_col="image")
s2_calib_df = calc_cmaps_df(s2_df, mask=s2_mask, image_col="image")

In [ ]:
def fill_frame(array, crop_region):
    x1, x2, y1, y2 = list(crop_region)
    return array[x1:x2, y1:y2]

In [ ]:
comparison_seqs = s2_calib_df["seq_name"].unique()

n_row = 2
n_img_col = len(comparison_seqs)   # dynamic number of image columns
n_col = n_img_col + 1              # +1 for colorbar column

# -------------------------
# Layout
# -------------------------
fig = plt.figure(figsize=(2 * n_img_col + 1, 5))

gs = fig.add_gridspec(
    n_row,
    n_col,
    width_ratios=[1] * n_img_col + [0.1],
    wspace=0.25,
    hspace=0.35
)

axes = [
    [fig.add_subplot(gs[i, j]) for j in range(n_img_col)]
    for i in range(n_row)
]

cax = fig.add_subplot(gs[:, -1])

centre = np.array(combo_mask.shape) // 2
dz = -3

vmin = 0.65
vmax = 1.35
slice_crop = [2, -2, 2, -3]

fs_title = 12

for i, seq in enumerate(comparison_seqs):

    seq_cropped = seq.replace("_scanner_comp", "")

    s1_subset_df = s1_calib_df[
        s1_calib_df["seq_name"] == seq_cropped
    ].copy()

    s2_subset_df = s2_calib_df[
        s2_calib_df["seq_name"] == seq
    ].copy()

    n_min = min(s1_subset_df.shape[0], s2_subset_df.shape[0])

    s1_subset_df = s1_subset_df.head(n_min)
    s2_subset_df = s2_subset_df.head(n_min)

    s1_cmap = np.mean(s1_subset_df["cmap"].values)
    s2_cmap = np.mean(s2_subset_df["cmap"].values)

    s1_cmap_bspline = cal.bspline_estimation(
        s1_cmap,
        s1_spacing,
        mask=combo_mask
    )

    s2_cmap_bspline = cal.bspline_estimation(
        s2_cmap,
        s2_spacing,
        mask=combo_mask
    )

    dw_axis = s1_subset_df["dwi_direction"].unique()[0]
    bval = int(np.round(s1_subset_df["b_value"].unique()[0], -2))

    # -------------------------
    # Slices
    # -------------------------
    s1_slice = s1_cmap_bspline[:, :, centre[2] + dz]
    s1_slice = fill_frame(np.rot90(s1_slice, k=1), slice_crop)

    s2_slice = s2_cmap_bspline[:, :, centre[2] + dz]
    s2_slice = fill_frame(np.rot90(s2_slice, k=1), slice_crop)

    mask_slice = combo_mask[:, :, centre[2] + dz]

    plot_mask = np.full_like(mask_slice, np.nan, dtype=float)
    plot_mask[mask_slice == 0] = 1
    plot_mask = fill_frame(np.rot90(plot_mask, k=1), slice_crop)

    # -------------------------
    # Plotting
    # -------------------------
    ax0 = axes[0][i]
    ax1 = axes[1][i]

    ax0.imshow(
        s1_slice,
        vmin=vmin,
        vmax=vmax,
        cmap=cc.cm.coolwarm,
        interpolation="nearest"
    )

    ax0.imshow(
        plot_mask,
        vmin=0,
        vmax=1,
        cmap=cm.Greys_r,
        interpolation="nearest"
    )

    ax0.set_title(
        rf"$b_{{{dw_axis}}} = {bval}$",
        fontsize=fs_title
    )

    im = ax1.imshow(
        s2_slice,
        vmin=vmin,
        vmax=vmax,
        cmap=cc.cm.coolwarm,
        interpolation="nearest"
    )

    ax1.imshow(
        plot_mask,
        vmin=0,
        vmax=1,
        cmap=cm.Greys_r,
        interpolation="nearest"
    )

# -------------------------
# Row labels
# -------------------------
for i in range(n_row):

    axes[i][0].text(
        -0.05,
        0.5,
        rf"$Scanner\ {i+1}$",
        transform=axes[i][0].transAxes,
        fontsize=fs_title,
        ha="right",
        va="center",
    )

# -------------------------
# Turn axes off
# -------------------------
for row in axes:
    for ax in row:
        ax.set_axis_off()

# -------------------------
# Colorbar
# -------------------------
cbar = fig.colorbar(im, cax=cax)

cbar.ax.set_title(
    r"$c_{Meas}$",
    fontsize=12,
    pad=10
)

cbar.ax.tick_params(labelsize=12)

plot_fname = output_dir / f"scanner_comparison_b_{bval:.0f}.png"

fig.savefig(
    fname=plot_fname,
    dpi=600,
    format="png",
    bbox_inches="tight",
    pad_inches=0.1)